# 03 — BGE-M3 Dense Embeddings

This notebook implements **Phase 04 only**. It loads the Phase 03 chunk dataset, defines an exact `BAAI/bge-m3` dense-embedding workflow with batching and explicit L2 normalization, preserves every chunk field, and writes a reusable embedding artifact when the model can run.

> **Scope boundary:** This notebook does not create Qdrant collections or implement retrieval, BM25, reranking, LLM calls, LangGraph, prompting, or chat functionality.

The official BGE-M3 model card documents 1,024-dimensional dense vectors and `BGEM3FlagModel` usage. [1] The exact model is required; this notebook never substitutes a smaller or different embedding model.

## Environment requirement

Run `pip install -U FlagEmbedding` in an environment with sufficient disk and memory before enabling the full embedding step. The official BGE-M3 weight file is approximately 2.27 GB; loading it safely requires additional runtime headroom. The preflight below blocks execution rather than downloading a model that the current environment cannot load. [1] [2]

In [1]:
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path
from typing import Any, Iterator

import numpy as np
from IPython.display import JSON, Markdown, display

MODEL_NAME = 'BAAI/bge-m3'
EXPECTED_DENSE_DIMENSION = 1024
MODEL_WEIGHT_BYTES = 2_271_145_830  # Official pytorch_model.bin size from the BAAI/bge-m3 file manifest.
MIN_RUNTIME_MEMORY_BYTES = MODEL_WEIGHT_BYTES * 2
BATCH_SIZE = 8
MAX_LENGTH = 1024
ENABLE_MODEL_EXECUTION = True
INPUT_NAME = 'introduction_to_business_chunks.jsonl'

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'data' / 'processed' / INPUT_NAME).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f'Could not find data/processed/{INPUT_NAME} from {Path.cwd()}')

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / INPUT_NAME
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
EMBEDDINGS_PATH = OUTPUT_DIR / 'introduction_to_business_bge_m3_embeddings.jsonl'
SUMMARY_PATH = OUTPUT_DIR / 'introduction_to_business_bge_m3_embedding_summary.json'
STATUS_PATH = OUTPUT_DIR / 'introduction_to_business_bge_m3_embedding_status.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Chunk input: {INPUT_PATH}')
print(f'Exact embedding model: {MODEL_NAME}')

Project root: /home/ubuntu/business-knowledge-ai
Chunk input: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_chunks.jsonl
Exact embedding model: BAAI/bge-m3


## 1. Load the chunk dataset and inspect provenance

The final embedding records will begin as copies of these chunk records. That preserves every inherited field in addition to the required `chunk_id`, `text`, `source`, `page`, `chapter`, and `section` fields.

In [2]:
with INPUT_PATH.open(encoding='utf-8') as handle:
    chunks = [json.loads(line) for line in handle if line.strip()]

required_chunk_fields = {'chunk_id', 'text', 'source', 'page', 'chapter', 'section'}
assert chunks, 'Phase 03 produced no chunks.'
assert all(required_chunk_fields <= chunk.keys() for chunk in chunks)
assert all(chunk['text'].strip() for chunk in chunks)

display(JSON({
    'chunk_count': len(chunks),
    'required_provenance_fields': sorted(required_chunk_fields),
    'sample_inputs': [
        {
            'chunk_id': chunk['chunk_id'],
            'page': chunk['page'],
            'chapter': chunk['chapter'],
            'section': chunk['section'],
            'text_preview': chunk['text'][:350],
        }
        for chunk in chunks[:3]
    ],
}))

<IPython.core.display.JSON object>

## 2. Model-resource preflight

This is intentionally conservative. The 2.27 GB BGE-M3 weights require more memory than the weights alone once loaded into a framework. If the host cannot safely load the exact model, the notebook writes a truthful status record and does not generate placeholder vectors.

In [3]:
def available_memory_bytes() -> int:
    try:
        with open('/proc/meminfo', encoding='utf-8') as handle:
            values = {line.split(':', 1)[0]: int(line.split()[1]) * 1024 for line in handle if ':' in line}
        return values.get('MemAvailable', 0)
    except FileNotFoundError:
        return 0

disk_free_bytes = shutil.disk_usage(PROJECT_ROOT).free
memory_free_bytes = available_memory_bytes()
preflight_reasons: list[str] = []
if not ENABLE_MODEL_EXECUTION:
    preflight_reasons.append('Model execution was explicitly disabled.')
if disk_free_bytes < MODEL_WEIGHT_BYTES:
    preflight_reasons.append('Insufficient free disk space for the exact model weights.')
if memory_free_bytes < MIN_RUNTIME_MEMORY_BYTES:
    preflight_reasons.append('Insufficient currently available memory for safe BGE-M3 loading with runtime headroom.')

preflight = {
    'model_name': MODEL_NAME,
    'model_weight_bytes': MODEL_WEIGHT_BYTES,
    'minimum_runtime_memory_bytes': MIN_RUNTIME_MEMORY_BYTES,
    'available_memory_bytes': memory_free_bytes,
    'free_disk_bytes': disk_free_bytes,
    'can_attempt_exact_model': not preflight_reasons,
    'reasons': preflight_reasons,
    'no_model_substitution': True,
}
display(JSON(preflight))

<IPython.core.display.JSON object>

## 3. Exact BGE-M3 batching and normalization implementation

The function below is the real `BGEM3FlagModel` dense-embedding path documented by BAAI. It processes chunks in batches, checks the reported vector dimension, computes raw L2 norms, normalizes every vector explicitly, and copies the original chunk record before adding embedding fields.

In [4]:
def batched(items: list[dict[str, Any]], batch_size: int) -> Iterator[list[dict[str, Any]]]:
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]

def generate_bge_m3_embeddings(records: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    from FlagEmbedding import BGEM3FlagModel

    model = BGEM3FlagModel(MODEL_NAME, use_fp16=False)
    embedded_records: list[dict[str, Any]] = []
    raw_norms: list[float] = []
    normalized_norms: list[float] = []

    for batch_index, batch in enumerate(batched(records, BATCH_SIZE), start=1):
        texts = [record['text'] for record in batch]
        output = model.encode(
            texts,
            batch_size=len(texts),
            max_length=MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        dense_vectors = np.asarray(output['dense_vecs'], dtype=np.float32)
        if dense_vectors.ndim != 2 or dense_vectors.shape[1] != EXPECTED_DENSE_DIMENSION:
            raise ValueError(f'Expected (*, {EXPECTED_DENSE_DIMENSION}) dense vectors; received {dense_vectors.shape}.')

        batch_raw_norms = np.linalg.norm(dense_vectors, axis=1)
        if np.any(batch_raw_norms == 0):
            raise ValueError('BGE-M3 returned a zero-norm vector, which cannot be normalized safely.')
        normalized_vectors = dense_vectors / batch_raw_norms[:, None]
        batch_normalized_norms = np.linalg.norm(normalized_vectors, axis=1)

        raw_norms.extend(float(value) for value in batch_raw_norms)
        normalized_norms.extend(float(value) for value in batch_normalized_norms)
        for record, vector in zip(batch, normalized_vectors, strict=True):
            embedded = dict(record)  # Preserve every Phase 03 metadata field.
            embedded.update({
                'embedding_model': MODEL_NAME,
                'embedding_dimension': int(vector.shape[0]),
                'embedding_normalization': 'l2',
                'embedding': vector.tolist(),
            })
            embedded_records.append(embedded)
        print(f'Completed batch {batch_index}: {len(batch)} chunks')

    diagnostics = {
        'embedding_dimension': EXPECTED_DENSE_DIMENSION,
        'batch_size': BATCH_SIZE,
        'raw_norm_example': raw_norms[:3],
        'normalized_norm_example': normalized_norms[:3],
        'normalized_norm_min': min(normalized_norms),
        'normalized_norm_max': max(normalized_norms),
    }
    return embedded_records, diagnostics

## 4. Execute when available; otherwise record the verified limitation

When preflight passes, this cell generates and persists only real BGE-M3 vectors. When it fails, it writes a truthful status JSON file, leaves the embedding artifact absent, and displays the reason. No alternate model or synthetic vectors are used.

In [5]:
execution_status: dict[str, Any] = {
    'model_name': MODEL_NAME,
    'input_artifact': str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    'embedding_artifact': str(EMBEDDINGS_PATH.relative_to(PROJECT_ROOT)),
    'no_model_substitution': True,
    'embeddings_generated': False,
}

if not preflight['can_attempt_exact_model']:
    execution_status.update({
        'status': 'blocked_by_resource_preflight',
        'limitation': 'The exact BAAI/bge-m3 model was not downloaded or loaded because this environment lacks safe runtime capacity.',
        'preflight': preflight,
    })
else:
    try:
        embedded_records, diagnostics = generate_bge_m3_embeddings(chunks)
        with EMBEDDINGS_PATH.open('w', encoding='utf-8') as handle:
            for record in embedded_records:
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')

        with EMBEDDINGS_PATH.open(encoding='utf-8') as handle:
            persisted = [json.loads(line) for line in handle if line.strip()]
        assert len(persisted) == len(chunks)
        assert all(required_chunk_fields <= record.keys() for record in persisted)
        assert all(record['embedding_model'] == MODEL_NAME for record in persisted)
        assert all(record['embedding_dimension'] == EXPECTED_DENSE_DIMENSION for record in persisted)

        summary = {
            'model_name': MODEL_NAME,
            'input_chunk_count': len(chunks),
            'embedding_record_count': len(persisted),
            'embedding_artifact': str(EMBEDDINGS_PATH.relative_to(PROJECT_ROOT)),
            'embedding_dimension': EXPECTED_DENSE_DIMENSION,
            'batch_size': BATCH_SIZE,
            'max_length': MAX_LENGTH,
            'normalization': 'l2',
            'diagnostics': diagnostics,
            'phase_scope': 'Dense BAAI/bge-m3 embedding generation only; no Qdrant, retrieval, BM25, reranking, LLM, or LangGraph.',
        }
        SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        execution_status.update({
            'status': 'completed',
            'embeddings_generated': True,
            'embedding_record_count': len(persisted),
            'summary_artifact': str(SUMMARY_PATH.relative_to(PROJECT_ROOT)),
            'example_embeddings': [
                {
                    'chunk_id': record['chunk_id'],
                    'embedding_dimension': record['embedding_dimension'],
                    'normalized_vector_preview': record['embedding'][:8],
                }
                for record in persisted[:3]
            ],
        })
    except Exception as error:
        execution_status.update({
            'status': 'model_load_or_encoding_failed',
            'limitation': 'The exact BAAI/bge-m3 model could not be loaded or used. No embeddings were written.',
            'error_type': type(error).__name__,
            'error_message': str(error),
        })

STATUS_PATH.write_text(json.dumps(execution_status, ensure_ascii=False, indent=2), encoding='utf-8')
display(JSON(execution_status))

<IPython.core.display.JSON object>

## Phase 04 result

If the execution status is `completed`, `data/processed/introduction_to_business_bge_m3_embeddings.jsonl` is the reusable dense-embedding artifact. If it is blocked or failed, use the recorded status to provision an environment capable of loading the exact model, then rerun this notebook.

### References

[1] https://huggingface.co/BAAI/bge-m3 — Official BAAI/bge-m3 model card.

[2] https://github.com/flagopen/FlagEmbedding — Official FlagEmbedding toolkit.